# Декораторы, итераторы и генераторы


Сегодня мы научимся добавлять функции новое поведение и обрабатывать данные по одному элементу.

## Замыкание

В Python можно объявить функцию внутри другой функции и потом ее использовать:

In [ ]:
def foo():
    def bar():
        print("Hi")
    bar()

foo()

Hi


Снаружи функции она будет недоступна:

In [ ]:
bar()

NameError: name 'bar' is not defined

Вложенная функция может использовать переменные внешней функции даже после завершения
её вызова. Функция с такими сохранёнными связями называется **замыканием**.


In [ ]:
factor = 10

def make_multiplier(len: int):
    def multiply(value: int):
        return value * len

    return multiply  # Возвращаем функцию, а не результат её вызова.


double = make_multiplier(2)
print(double)
triple = make_multiplier(3)
print(triple)
print(double(10))  # 20
# print(triple(10))  # 30


<function make_multiplier.<locals>.multiply at 0x7afb93b11760>
<function make_multiplier.<locals>.multiply at 0x7afb93b11940>
20


In [ ]:
def foo(fn):
    for i in range(10):
      print(fn(i))

foo(triple)

0
3
6
9
12
15
18
21
24
27


In [ ]:
def counter(intitial: int = 2):
    def increment():
        nonlocal intitial
        intitial += 1
        return intitial
    return increment

c = counter()
print(c())
print(c())

c2 = counter(10)
print(c2())

3
4
11


In [ ]:
def collect():
    numbers = []
    def inner(number: int) -> list[int]:
        numbers.append(number)
        if len(numbers) > 10:
            numbers.clear()
            numbers.append(number)
        return numbers
    return inner

c = collect()
c2 = collect()

c(10)
c(10)
c(10)
c(10)
c(10)
c(10)
c(10)
c(10)
c(10)
print(c(10))
print(c(10))
c.numbers

[10, 10, 10, 10, 10, 10, 10, 10, 10, 10]
[10]


[10]

In [ ]:
def relative_max():
    prev_max = float("-inf")
    def compare(value: int) -> int:
        nonlocal prev_max
        if value > prev_max:
            prev_max = value
        return prev_max
    return compare


comparer = relative_max()
comparer2 = relative_max()

comparer(10)
comparer(5)
comparer(20)

comparer2(-1)
comparer2(25)

In [ ]:
class RelativeMax
    def __init__(self):
        self.prev_max = float("-inf")

    def compare(self, value: int) -> int:
        if value > self.prev_max:
            self.prev_max = value
        return self.prev_max

comparer = RelativeMax()
comparer.compare(5)
comparer.compare(3)
comparer.compare(18)

Какой результат будет у такого вызова?

In [ ]:
make_multiplier(2)(3)

Будет ли переменная factor для функции multiply локальной или глобальной? Ни то, ни другое!

В аббревиатуре **LEGB** такая переменная будет **E** - *enclosing*, замкнутая переменная.

Можем ли мы ее модифицировать?

In [ ]:
def foo(x):
    def bar(y):
        x = x + y
        print(x)
    return bar

foo(10)(20)

В Python есть инструкция `nonlocal` - работает по аналогии с `global`, но для замкнутых переменных:

In [ ]:
def foo(x):
    def bar(y):
        nonlocal x
        x = x + y
        print(x)
    return bar

foo(10)(20)

## Декораторы


Представим задачу. Нам нужно выводить имя вызываемой функции, сохранив её основную логику.
Если добавить `print()` в каждую функцию, код будет повторяться.

**Декоратор** принимает функцию и возвращает объект, который будет использоваться вместо неё.
В данном случае этот объект — функция-обёртка. Она добавляет действия до или после исходного вызова.


In [ ]:
def announce(func):
    def wrapper(*args, **kwargs):
        print(f"Вызываем {func.__name__}")
        return func(*args, **kwargs)

    return wrapper


def add(a, b):
    return a + b


add = announce(add)
print(add(2, b=3))  # Вызываем add, затем 5


Вызываем add
5


В этом примере важно различать три действия:

1. `announce(add)` создаёт обёртку; тело `add` пока не запускается.
2. `return wrapper` возвращает саму обёртку.
3. Последующий `add(2, b=3)` запускает обёртку, а та — исходную функцию.

`*args` собирает позиционные аргументы, `**kwargs` — именованные.
В вызове `func(*args, **kwargs)` они распаковываются обратно.
`return` передаёт результат исходной функции вызывающему коду.
Без него результатом обёртки будет `None`.

`@announce` над определением эквивалентно присваиванию
`multiply = announce(multiply)` после определения функции.


In [ ]:
@announce
def multiply(a, b=2):
    """Возвращает произведение двух чисел."""
    return a * b


print(multiply(4, b=3))  # 12
print(multiply.__name__)  # wrapper — метаданные потерялись


Вызываем multiply
12
wrapper


Имя `multiply` теперь указывает на обёртку, поэтому без дополнительных действий её `__name__` и документация будут описывать `wrapper`.

`@wraps(func)` переносит основные метаданные (имя, аннотации типов, документацию) исходной функции на обёртку.


In [ ]:
from functools import wraps


def announce(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"Вызываем {func.__name__} с аргументами {args} и {kwargs}")
        return func(*args, **kwargs)

    return wrapper


@announce
def subtract(a, b=1):
    """Вычитает b из a."""
    return a - b



print(subtract(10, b=3))  # 7
print(subtract(5, 2))  # 7
print(subtract.__name__)  # subtract
print(subtract.__doc__)  # Вычитает b из a.


Вызываем subtract с аргументами (10,) и {'b': 3}
7
Вызываем subtract с аргументами (5, 2) и {}
3
subtract
Вычитает b из a.


Существующая `multiply` не изменилась: она была обёрнута предыдущей версией
`announce`. Новая версия применяется при последующем декорировании.

Декораторы полезны для множества операций, самые явные примеры - логирование, измерение времени. Они могут менять поведение, но наш декоратор сохраняет результат и не перехватывает исключения исходной функции.


Задание:

Напишем декоратор `trace`, который:

- печатает `Начало: имя` перед вызовом;
- вызывает исходную функцию ровно один раз, передавая все аргументы;
- после успешного вызова печатает `Результат: значение`;
- возвращает результат и сохраняет имя и документацию через `wraps`.

Если исходная функция выбрасывает исключение, оно должно дойти до вызывающего кода.
Сообщение о результате в таком случае не печатается.


In [ ]:
def trace(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        # TODO: сообщение, вызов, сообщение, return
        raise NotImplementedError("Допишите wrapper")

    return wrapper


@trace
def power(base, exponent=2):
    """Возводит base в степень exponent."""
    return base ** exponent

assert power(3, exponent=2) == 9
assert power.__name__ == "power"
assert power.__doc__ == "Возводит base в степень exponent."


In [11]:
import time
from functools import cache

def decorator(func):  # func = multiply
    def f(*args, **kwargs):  # args=(10, 20), kwargs={}
        start = time.time_ns()
        result = func(*args, **kwargs)  # multiply(*(10, 20)) -> multiply(10, 20)
        # result = 200
        end = time.time_ns()
        total = end - start
        print(f"Функция {func.__name__} работала {total}")
        return result
    return f


@decorator
@cache
def multiply(a, b):
    return a * b

# def multiply(*args, **kwargs):
#     start = time.time()
#     result = original_multiply(*args, **kwargs)
#     end = time.time()
#     total = end - start
#     print(f"Функция multiply работала {total}")
#     return result

@decorator
def divide(a, b):
    return a / b

# def divide(*args, **kwargs):
#     start = time.time()
#     result = original_divide(*args, **kwargs)
#     end = time.time()
#     total = end - start
#     print(f"Функция divide работала {total}")
#     return result

multiply(10, 20)
multiply(10, 20)
multiply(10, 20)
multiply(20, 30)


def sub(a, b):
    return a - b


decorated_sub = decorator(sub)

Функция multiply работала 5000162169
Функция multiply работала 1691
Функция multiply работала 603
Функция multiply работала 5000144681


600

In [ ]:
def stupid_decorator(func):
    def f(a):
       return a
    return f

@stupid_decorator
def some_foo(a, b, c, d, e):
    return a * b + c / d ** e

some_foo(1)

In [12]:
def foo():
    pass


foo.a = 10
print(foo.a)

10


In [13]:
def some_deco(f):
    f.a = 10
    return f


@some_deco
def bar():
    pass


bar.a

10

In [15]:
REGISTRY = []

def register(f):
    if f not in REGISTRY:
        REGISTRY.append(f.__name__)
    return f


@register
def baz():
    pass

print(REGISTRY)

['baz']


## Итерируемые объекты и итераторы


Цикл `for` умеет работать со списком, строкой и файлом. Ему не нужно знать,
как именно эти объекты хранят данные: достаточно общего протокола обхода.

Существует подход - ленивые вычисления - когда мы не получаем новые данные из объекта до тех пор, пока они нам не нужны. В Python такой подход помогает реализовать функция `iter()` - она получает новое значение только тогда, когда потребуется (когда она будет вызвана).

**Итерируемый объект (iterable)** — объект, из которого `iter(obj)` может получить итератор.
**Итератор (iterator)** — объект, который хранит состояние обхода и выдаёт следующий
элемент по `next(iterator)`. Когда элементов больше нет, он поднимает `StopIteration`.


In [16]:
names = ["Аня", "Борис", "Вера"]
cursor = iter(names)
print(cursor)

print(next(cursor))  # Аня
print(next(cursor))  # Борис
print(next(cursor))  # Вера

try:
    next(cursor)
except StopIteration:
    print("Обход завершён")


Аня
Борис
Вера
Обход завершён


In [17]:
lst = [1, 2, 3]
next(lst)

TypeError: 'list' object is not an iterator

Список — итерируемый объект, но сам не итератор: `next(names)` вызвал бы `TypeError`.
У списка можно получать независимые итераторы. У итератора `iter()` возвращает его самого.


In [ ]:
names = ["Аня", "Борис", "Вера"]

left = iter(names)
right = iter(names)

print(next(left))   # Аня
print(next(left))   # Борис
print(next(right))  # Аня — собственная позиция
print(iter(left) is left)  # True


In [19]:
lst = [1, 2, 3, 4, 5]
rev = reversed(lst)
print(rev)
print(next(rev))

5


In [22]:
r = range(10)
next(r)

TypeError: 'range' object is not an iterator

In [38]:
from itertools import cycle

c = cycle("ABC")
next(c)
next(c)
next(c)
next(c)

'A'

In [ ]:
list(cycle("ABC"))

Как работает `for`?

Упрощённая модель цикла: один раз получить итератор, затем запрашивать элементы
до `StopIteration`. Для обычного кода используйте `for`; ручной цикл нужен здесь
для понимания протокола.


In [ ]:
cursor = iter([10, 20, 30])
while True:
    try:
        item = next(cursor)
    except StopIteration:
        break
    print(item)


`for`, `list()` и `sum()` потребляют элементы из текущей позиции итератора.
Они не возвращают его к началу. После исчерпания итератор остаётся исчерпанным.


In [26]:
cursor = iter([10, 20, 30])
print(next(cursor))      # 10
print(list(cursor))      # [20, 30]
print(list(cursor))      # []
print(next(cursor, "конец"))  # конец: значение по умолчанию вместо StopIteration


10
[20, 30]
[]
конец


Почему два вызова `list(names)` дадут одинаковые списки,
а два вызова `list(cursor)` могут дать разные результаты?


In [27]:
cursor = iter([10, 20, 30])
print(list(cursor))
print(list(cursor))

[10, 20, 30]
[]



У списка каждый раз создаётся новый итератор; в `cursor` продолжается уже начатый обход.
`range` тоже является итерируемым объектом, а не итератором: для него можно создавать
новые обходы, хотя он не хранит список всех своих чисел.


Итератор можно реализовать в виде класса, главное соблюсти *протокол* - реализовать нужные методы.

Протокол итератора состоит из двух методов:
- `.__iter__()` возвращает сам итератор;
- `.__next__()` возвращает очередное значение или поднимает `StopIteration`.

Наследоваться от специального класса для этого не обязательно — достаточно поведения.


In [37]:
class CountdownIterator:
    def __init__(self, start: int):
        self.current = start

    def __iter__(self):
        print("Здесь!")
        return self

    def __next__(self):
        if self.current <= 0:
            raise StopIteration
        value = self.current
        self.current -= 1
        return value


countdown_cursor = CountdownIterator(3)
print("Еще не здесь")
countdown_iter = iter(countdown_cursor)
print(next(countdown_iter))  # 3
print(list(countdown_iter))  # [2, 1]
print(list(countdown_iter))  # []
print(next(countdown_iter, 10_000))  # 10_000



Еще не здесь
Здесь!
3
Здесь!
[2, 1]
Здесь!
[]
10000


In [40]:
lst = [1, 2, 3]
cursor = iter(lst)
print(type(cursor))
print(cursor.__next__)

<class 'list_iterator'>
<method-wrapper '__next__' of list_iterator object at 0x780cd7c887f0>


In [39]:
countdown_cursor = CountdownIterator(3)
for elem in countdown_cursor:
    print(elem)

Здесь!
3
2
1


`return self` - объект возвращает самого себя, что нам подходит - дальше будет вызываться `.__next__()` этого же объекта.

Если нужен повторный обход - отделим описание последовательности от позиции обхода!

Объект `Countdown` будет хранить начало, а каждый вызов его `__iter__()` создаст новый итератор:


In [41]:
class Countdown:
    def __init__(self, start):
        self.start = start

    def __iter__(self):
        return CountdownIterator(self.start)


countdown_values = Countdown(3)
print(list(countdown_values))  # [3, 2, 1]
print(list(countdown_values))  # [3, 2, 1]

first = iter(countdown_values)
second = iter(countdown_values)
print(next(first), next(second))  # 3 3
assert list(Countdown(0)) == []


[3, 2, 1]
[3, 2, 1]
3 3


In [42]:
lst = [1, 2, 3]
first = iter(lst)
second = iter(lst)
print(next(first), next(second))

1 1


## Генераторы


**Генераторная функция** - функция, которая содержит инструкцию `yield`.

Её вызов создаёт **объект-генератор**, который является итератором. Тело функции начинает выполняться при первом запросе элемента.

`yield` выдаёт значение и приостанавливает выполнение, сохраняя локальные переменные
и место продолжения. Следующий `next()` продолжает работу после этого `yield`.
`return` или конец тела завершает генератор; вызывающая сторона получает `StopIteration`.
Внутри генератора не нужно писать `raise StopIteration`: используйте `return`.


In [43]:
def countdown(start):
    print("Генератор начал работу")
    current = start
    while current > 0:
        yield current
        current -= 1


generator = countdown(3)
print("Объект создан")  # Пока только эта строка
print(next(generator))  # Генератор начал работу, затем 3
print(next(generator))  # 2
print(list(generator))  # [1]
print(list(generator))  # []


Объект создан
Генератор начал работу
3
2
[1]
[]


In [66]:
def gen():
    print("Начали")
    yield 1
    print("Первый")
    yield 1
    print("Второй")
    yield 1
    print("Третий")
    yield 1
    print("Четвертый")
    yield 1
    print("Конец")

g = gen()

In [65]:
try:
    next(g)
except StopIteration:
    pass

In [67]:
for i in g:
  print(i)

Начали
1
Первый
1
Второй
1
Третий
1
Четвертый
1
Конец


Тот же отсчёт потребовал меньше кода: Python реализует `.__iter__()`, `.__next__()`
и хранит состояние выполнения. Чтобы начать заново, вызовите `countdown(3)` ещё раз.

Генераторные выражения позволяют создавыть быстрые генераторы. Они выглядят как list comprehension, но с круглыми скобками:


In [68]:
squares_list = [number ** 2 for number in range(5)]
squares_generator = (number ** 2 for number in range(5))

print(squares_list)             # [0, 1, 4, 9, 16]
print(next(squares_generator))  # 0
print(sum(squares_generator))   # 30: сумма оставшихся 1, 4, 9, 16
print(sum(squares_generator))   # 0: элементов больше нет


[0, 1, 4, 9, 16]
0
30
0


Что даёт ленивость?

Потребитель управляет тем, сколько элементов будет вычислено. Покажем это явно:


In [70]:
def range_with_print(limit):
    for number in range(limit):
        print(f"Получаем {number}")
        yield number

lazy_squares = (number ** 2 for number in range_with_print(5))

In [76]:
print("Запрашиваем один квадрат")
print(next(lazy_squares))  # Получаем 0, затем 0; числа 1–4 пока не запрашивались

Запрашиваем один квадрат


StopIteration: 

Генератор позволяет не хранить весь результат одновременно. Если исходный список
уже существует, его память от создания генератора не освободится. `list(generator)`
снова накопит все результаты; `sorted()` тоже материализует данные.
Ленивость сама по себе не обещает ускорения.

`sum()` потребляет поток до конца, но не создаёт список всех значений.
Генератору недоступны обычная индексация и `len()`.

Решим задачу.

Есть строки с целыми суммами расходов. Нужно пропустить пустые строки и комментарии,
преобразовать остальные строки в числа и посчитать сумму **только положительных** значений.
Некорректные числа по условию не встречаются; ошибки преобразования не перехватываем.

Напишем генераторную функцию `parse_amounts(lines)`, которая:

1. Перебирает строки по одной и применяет `.strip()`.
2. Пропускает пустые строки и строки, начинающиеся с `#`.
3. Выдаёт `int(text)` через `yield`.


In [78]:
raw_lines = ["120\n", "\n", " # возврат\n", "-20\n", " 80 \n", "0\n"]


def parse_amounts(lines):
    for line in lines:
        line = line.strip()  # "120"
        if not line or line.startswith("#"):
            continue
        yield int(line)

amounts = parse_amounts(raw_lines)
positive_amounts = (num for num in amounts if num >= 0)  # Напишем генераторное выражение
print(positive_amounts)
print(type(positive_amounts))
total = sum(positive_amounts)
print(total)  # 200
assert total == 200
assert list(positive_amounts) == []
no_amounts = parse_amounts(["\n", "# нет данных\n"])
assert list(no_amounts) == []


<generator object <genexpr> at 0x780cd7532670>
<class 'generator'>
200


In [ ]:
def sum_(iterable):
    acc = 0
    it = iter(iterable)
    while True:
        try:
            acc += next(it)
        except StopIteration:
            return acc



## Приватные методы

In [84]:
from ast import Attribute
class X:
    def __init__(self):
        self._x = 10

    @property
    def x(self):
        return self._x

    @x.setter
    def x(self, value):
        if value < 0:
            raise ...
        self._x = value

    def __method(self):
        pass
    def foo(self):
        return self.__method()
    def _hidden(self):
        pass

x = X()
try:
    x.__method()
except AttributeError:
    pass
x._X__method()